# 🛰️ Exploratory Data Analysis: Karhutla Kalimantan Tengah (2019-2023)
**Investigasi Spasiotemporal Titik Api, Interaksi Gambut-Kanal, dan Anomali Iklim El Niño**

Lingkungan Conda yang digunakan:
```bash
conda activate karhutla_env
```

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# Konfigurasi visualisasi
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("../data/processed/karhutla_kalteng_2019_2023.csv")
df["acq_date"] = pd.to_datetime(df["acq_date"])
print(f"Total baris data: {len(df):,}")
df.head()

## 1. Analisis Temporal: Tren Tahunan & Siklus El Niño

In [ ]:
yearly = df.groupby("year").agg(total_hotspot=("id", "count"), total_frp=("frp_mw", "sum")).reset_index()
plt.figure(figsize=(9, 4.5))
bars = plt.bar(yearly["year"], yearly["total_hotspot"], color=["#d73027", "#4575b4", "#74add1", "#abd9e9", "#f46d43"], edgecolor="black")
plt.title("Sebaran Hotspot per Tahun: Dominasi Tahun El Niño (2019 & 2023)", fontsize=12, fontweight="bold")
plt.xlabel("Tahun")
plt.ylabel("Jumlah Hotspot")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 100, f"{h:,}", ha="center", fontweight="bold")
plt.show()

## 2. Analisis Spasial: Episentrum Kabupaten Terparah

In [ ]:
kab = df["kabupaten_kota"].value_counts().reset_index()
kab.columns = ["kabupaten", "jumlah_hotspot"]
plt.figure(figsize=(10, 5))
sns.barplot(data=kab, x="jumlah_hotspot", y="kabupaten", palette="Reds_r")
plt.title("Peringkat Kabupaten/Kota dengan Hotspot Tertinggi di Kalteng", fontweight="bold")
plt.xlabel("Jumlah Titik Api")
plt.ylabel("")
plt.show()

## 3. Analisis Biofisik: Pengaruh Lahan Gambut & Jarak Kanal Drainase

In [ ]:
gambut_df = df[df["tipe_lahan"].str.contains("Gambut")]
plt.figure(figsize=(10, 4.5))
sns.histplot(gambut_df["jarak_kanal_m"], bins=35, kde=True, color="#b2182b")
plt.axvline(1000, color="orange", linestyle="--", label="Ambang Kritis 1.000 m")
plt.title("Distribusi Jarak ke Kanal Drainase pada Lahan Gambut", fontweight="bold")
plt.xlabel("Jarak ke Kanal (meter)")
plt.legend()
plt.show()

## 4. Pemodelan Machine Learning (Validasi Temporal Out-of-Time: 2019-2023 -> 2024-2026)
Membangun model ensemble untuk mengestimasi risiko kebakaran berkekuatan ekstrem.

In [ ]:
from src.train_model import train_karhutla_model
metrics = train_karhutla_model("../data/processed/karhutla_kalteng_2019_2026.csv", output_fig_dir="../output/figures", output_report_dir="../output/reports")
print("Hasil Pelatihan Model:", metrics)